# 2.3 Vectorization Techniques and Model Comparison

**Google Colab Ready** – optimized for GPU acceleration.

This notebook compares two vectorization pipelines for Vietnamese emotion classification:

1. **TF-IDF** → Traditional ML (Logistic Regression, Random Forest, SVM, Naive Bayes)
2. **Tokenization + Padding** → Deep Learning (LSTM, CNN)

### How to Use
1. Upload training data to Google Drive at: `MyDrive/thesis/data/`
   - Required files: `train_1500_para_final.csv`, `val_processed.csv`, `test_processed.csv`
2. Enable GPU in Colab: **Runtime → Change runtime type → GPU**
3. Run cells in order

---

## 0. Google Colab Setup & Imports

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("TPU detected and configured")
except ImportError:
    TPU_AVAILABLE = False

gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f"GPU detected: {gpu_info.stdout.strip()}")
else:
    print("No NVIDIA GPU detected – training will be slow on CPU.")

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn tensorflow keras

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Conv1D,
    GlobalMaxPooling1D,
    Bidirectional,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── GPU configuration ──
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"GPU(s) detected: {[g.name for g in gpus]}")
    print(f"Mixed-precision policy: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("No GPU detected – running on CPU.")

print(f"TensorFlow version: {tf.__version__}")

# ── Batch sizes ──
TRAIN_BATCH_SIZE = 128
VAL_BATCH_SIZE   = 264
print(f"Train batch size : {TRAIN_BATCH_SIZE}")
print(f"Val/Test batch   : {VAL_BATCH_SIZE}")

## 1. Data Loading

Load the pre-processed train / validation / test splits and normalise column names.

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/thesis/data")

TRAIN_FILE = DATA_DIR / "train_1500_para_final.csv"
VAL_FILE   = DATA_DIR / "val_processed.csv"
TEST_FILE  = DATA_DIR / "test_processed.csv"

for f in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    assert f.exists(), f"Missing: {f}"
print(f"Data directory: {DATA_DIR}")

VN_TO_EN = {
    "tức giận": "Anger",
    "khó chịu": "Disgust",
    "vui vẻ": "Enjoyment",
    "sợ hãi": "Fear",
    "khác": "Other",
    "buồn": "Sadness",
    "ngạc nhiên": "Surprise",
}


def load_and_normalise(path: Path) -> pd.DataFrame:
    """Load CSV and normalise to columns: text, label."""
    df = pd.read_csv(path)

    # Pick the best available text column
    if "Sentence_clean" in df.columns:
        df["text"] = df["Sentence_clean"]
    elif "Sentence" in df.columns:
        df["text"] = df["Sentence"]
    else:
        raise ValueError(f"No text column found in {path}")

    # Pick the best available label column
    if "Emotion" in df.columns:
        df["label"] = df["Emotion"]
    elif "emotion_vn" in df.columns:
        df["label"] = df["emotion_vn"].map(VN_TO_EN)
    else:
        raise ValueError(f"No label column found in {path}")

    df = df[["text", "label"]].dropna()
    return df


train_df = load_and_normalise(TRAIN_FILE)
val_df   = load_and_normalise(VAL_FILE)
test_df  = load_and_normalise(TEST_FILE)

print(f"Train : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")
print(f"\nLabels: {sorted(train_df['label'].unique())}")
train_df.head()

In [ ]:
le = LabelEncoder()
le.fit(sorted(train_df["label"].unique()))

y_train = le.transform(train_df["label"])
y_val   = le.transform(val_df["label"])
y_test  = le.transform(test_df["label"])

num_classes = len(le.classes_)
print(f"Number of classes: {num_classes}")
print(f"Class mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

### Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df_) in zip(axes, [("Train", train_df), ("Val", val_df), ("Test", test_df)]):
    df_["label"].value_counts().sort_index().plot.bar(ax=ax, color="steelblue", edgecolor="black")
    ax.set_title(f"{name} – Label Distribution (n={len(df_)})")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

---

## 2. Traditional Machine Learning Pipeline

### 2.1 Vectorization: TF-IDF (Term Frequency – Inverse Document Frequency)

`TfidfVectorizer(max_features=5000)` converts raw text into fixed-length numerical vectors
where each dimension corresponds to a word weighted by its importance across the corpus.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(train_df["text"])
X_val_tfidf   = tfidf.transform(val_df["text"])
X_test_tfidf  = tfidf.transform(test_df["text"])

print(f"TF-IDF vocabulary size : {len(tfidf.vocabulary_)}")
print(f"X_train_tfidf shape    : {X_train_tfidf.shape}")
print(f"X_val_tfidf shape      : {X_val_tfidf.shape}")
print(f"X_test_tfidf shape     : {X_test_tfidf.shape}")

### 2.2 Model Training & Evaluation

In [ ]:
ml_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, C=1.0, solver="lbfgs", multi_class="multinomial", random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1
    ),
    "SVM (LinearSVC)": LinearSVC(
        C=1.0, max_iter=2000, random_state=SEED
    ),
    "Naive Bayes": MultinomialNB(alpha=1.0),
}

ml_results = []

for name, model in ml_models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    model.fit(X_train_tfidf, y_train)

    y_val_pred  = model.predict(X_val_tfidf)
    y_test_pred = model.predict(X_test_tfidf)

    val_acc  = accuracy_score(y_val, y_val_pred)
    val_f1   = f1_score(y_val, y_val_pred, average="weighted")
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1  = f1_score(y_test, y_test_pred, average="weighted")

    print(f"\nValidation  – Accuracy: {val_acc:.4f}  |  F1 (weighted): {val_f1:.4f}")
    print(f"Test        – Accuracy: {test_acc:.4f}  |  F1 (weighted): {test_f1:.4f}")
    print(f"\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred, target_names=le.classes_))

    ml_results.append({
        "Model": name,
        "Vectorization": "TF-IDF",
        "Val Accuracy": val_acc,
        "Val F1": val_f1,
        "Test Accuracy": test_acc,
        "Test F1": test_f1,
    })

### 2.3 Confusion Matrices – Traditional ML

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

for ax, (name, model) in zip(axes, ml_models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
    )
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices – TF-IDF + Traditional ML", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Deep Learning Pipeline

### 3.1 Vectorization: Tokenization & Padding

`Tokenizer(num_words=max_words)` builds a word-index dictionary from the training corpus.
`texts_to_sequences` maps each sentence to a list of integer IDs.
`pad_sequences` ensures all sequences have the same length for batch processing.

In [ ]:
MAX_WORDS   = 5000
MAX_SEQ_LEN = 128
EMBED_DIM   = 128

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train_seq = tokenizer.texts_to_sequences(train_df["text"])
X_val_seq   = tokenizer.texts_to_sequences(val_df["text"])
X_test_seq  = tokenizer.texts_to_sequences(test_df["text"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_SEQ_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_SEQ_LEN, padding="post", truncating="post")

y_train_cat = to_categorical(y_train, num_classes)
y_val_cat   = to_categorical(y_val,   num_classes)
y_test_cat  = to_categorical(y_test,  num_classes)

vocab_size = max(MAX_WORDS, len(tokenizer.word_index) + 1)

print(f"Tokenizer word_index size: {len(tokenizer.word_index)}")
print(f"Embedding vocab_size     : {vocab_size}")
print(f"Max sequence length      : {MAX_SEQ_LEN}")
print(f"X_train_pad shape        : {X_train_pad.shape}")
print(f"X_val_pad shape          : {X_val_pad.shape}")
print(f"X_test_pad shape         : {X_test_pad.shape}")
print(f"y_train_cat shape        : {y_train_cat.shape}")

### 3.2 Sequence Length Distribution

In [ ]:
seq_lengths = [len(s) for s in X_train_seq]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths, bins=50, color="steelblue", edgecolor="black", alpha=0.8)
ax.axvline(MAX_SEQ_LEN, color="red", linestyle="--", label=f"MAX_SEQ_LEN={MAX_SEQ_LEN}")
ax.axvline(np.median(seq_lengths), color="orange", linestyle="--", label=f"Median={np.median(seq_lengths):.0f}")
ax.set_title("Training – Sequence Length Distribution")
ax.set_xlabel("Sequence Length (tokens)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

pct_within = np.mean([l <= MAX_SEQ_LEN for l in seq_lengths]) * 100
print(f"{pct_within:.1f}% of training sequences fit within MAX_SEQ_LEN={MAX_SEQ_LEN}")

### 3.3 LSTM Model

In [ ]:
def build_lstm(vocab_size, embed_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=True)),
        Bidirectional(LSTM(32)),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, dtype="float32", activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


lstm_model = build_lstm(vocab_size, EMBED_DIM, MAX_SEQ_LEN, num_classes)
lstm_model.summary()

In [ ]:
EPOCHS = 30

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
]

lstm_history = lstm_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    validation_batch_size=VAL_BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
def plot_training_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history["accuracy"], label="Train")
    ax1.plot(history.history["val_accuracy"], label="Val")
    ax1.set_title(f"{title} – Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()

    ax2.plot(history.history["loss"], label="Train")
    ax2.plot(history.history["val_loss"], label="Val")
    ax2.set_title(f"{title} – Loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend()

    plt.tight_layout()
    plt.show()


plot_training_history(lstm_history, "Bi-LSTM")

In [ ]:
def evaluate_dl_model(model, X_pad, y_true, le, model_name):
    y_prob = model.predict(X_pad, batch_size=VAL_BATCH_SIZE, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="weighted")

    print(f"\n{model_name} – Test Results")
    print(f"Accuracy: {acc:.4f}  |  F1 (weighted): {f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_))

    return acc, f1, y_pred


lstm_test_acc, lstm_test_f1, lstm_preds = evaluate_dl_model(
    lstm_model, X_test_pad, y_test, le, "Bi-LSTM"
)

lstm_val_loss, lstm_val_acc = lstm_model.evaluate(X_val_pad, y_val_cat, batch_size=VAL_BATCH_SIZE, verbose=0)
lstm_val_f1 = f1_score(y_val, np.argmax(lstm_model.predict(X_val_pad, batch_size=VAL_BATCH_SIZE, verbose=0), axis=1), average="weighted")

### 3.4 CNN Model

In [ ]:
def build_cnn(vocab_size, embed_dim, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, embed_dim, input_length=max_len),
        Conv1D(128, kernel_size=5, activation="relu"),
        GlobalMaxPooling1D(),
        Dense(128, activation="relu"),
        Dropout(0.4),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, dtype="float32", activation="softmax"),
    ])
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


cnn_model = build_cnn(vocab_size, EMBED_DIM, MAX_SEQ_LEN, num_classes)
cnn_model.summary()

In [ ]:
cnn_history = cnn_model.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=TRAIN_BATCH_SIZE,
    validation_batch_size=VAL_BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

plot_training_history(cnn_history, "CNN")

In [ ]:
cnn_test_acc, cnn_test_f1, cnn_preds = evaluate_dl_model(
    cnn_model, X_test_pad, y_test, le, "CNN"
)

cnn_val_loss, cnn_val_acc = cnn_model.evaluate(X_val_pad, y_val_cat, batch_size=VAL_BATCH_SIZE, verbose=0)
cnn_val_f1 = f1_score(y_val, np.argmax(cnn_model.predict(X_val_pad, batch_size=VAL_BATCH_SIZE, verbose=0), axis=1), average="weighted")

### 3.5 Confusion Matrices – Deep Learning

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, preds) in zip(axes, [("Bi-LSTM", lstm_preds), ("CNN", cnn_preds)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Oranges",
        xticklabels=le.classes_, yticklabels=le.classes_, ax=ax,
    )
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices – Tokenization+Padding + Deep Learning", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 4. Overall Model Comparison

In [ ]:
dl_results = [
    {
        "Model": "Bi-LSTM",
        "Vectorization": "Tokenization + Padding",
        "Val Accuracy": lstm_val_acc,
        "Val F1": lstm_val_f1,
        "Test Accuracy": lstm_test_acc,
        "Test F1": lstm_test_f1,
    },
    {
        "Model": "CNN",
        "Vectorization": "Tokenization + Padding",
        "Val Accuracy": cnn_val_acc,
        "Val F1": cnn_val_f1,
        "Test Accuracy": cnn_test_acc,
        "Test F1": cnn_test_f1,
    },
]

all_results = pd.DataFrame(ml_results + dl_results)
all_results = all_results.sort_values("Test F1", ascending=False).reset_index(drop=True)

for col in ["Val Accuracy", "Val F1", "Test Accuracy", "Test F1"]:
    all_results[col] = all_results[col].map(lambda x: round(x, 4))

print("\n" + "=" * 80)
print("  MODEL COMPARISON – Sorted by Test F1 (weighted)")
print("=" * 80)
all_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ["#4C72B0" if v == "TF-IDF" else "#DD8452" for v in all_results["Vectorization"]]

axes[0].barh(all_results["Model"], all_results["Test Accuracy"], color=colors, edgecolor="black")
axes[0].set_xlabel("Test Accuracy")
axes[0].set_title("Test Accuracy by Model")
axes[0].set_xlim(0, 1)
for i, v in enumerate(all_results["Test Accuracy"]):
    axes[0].text(v + 0.01, i, f"{v:.4f}", va="center")

axes[1].barh(all_results["Model"], all_results["Test F1"], color=colors, edgecolor="black")
axes[1].set_xlabel("Test F1 (weighted)")
axes[1].set_title("Test F1 (weighted) by Model")
axes[1].set_xlim(0, 1)
for i, v in enumerate(all_results["Test F1"]):
    axes[1].text(v + 0.01, i, f"{v:.4f}", va="center")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4C72B0", edgecolor="black", label="TF-IDF (Traditional ML)"),
    Patch(facecolor="#DD8452", edgecolor="black", label="Tokenization+Padding (Deep Learning)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=2, fontsize=11, bbox_to_anchor=(0.5, -0.05))

plt.suptitle("Vectorization Technique Comparison", fontsize=14)
plt.tight_layout()
plt.show()

---

## 5. Summary

| Pipeline | Vectorization | Models | Key Characteristic |
|---|---|---|---|
| Traditional ML | TF-IDF (`max_features=5000`) | LR, RF, SVM, NB | Bag-of-words; fast training; no sequence awareness |
| Deep Learning | Tokenization + Padding (`num_words=5000`, `maxlen=128`) | Bi-LSTM, CNN | Learns word embeddings; captures sequential / local patterns |